# Prerequisite · Topic 0 — Environments & Conda

---

## 1. What it is

A **virtual environment** is an isolated, self-contained folder that holds its own copy of the Python interpreter and its own installed packages, walled off from the system Python and from every other environment. One project = one environment; environments can hold conflicting package versions without ever colliding.

The problem it solves is **dependency hell**: a single machine has only one system Python, so it can hold only one version of any given package. If Project A needs `numpy 1.24` and Project B needs `numpy 2.0`, they cannot coexist in one Python — installing for one breaks the other. Isolated environments give each project its own package set.

**Conda** does two jobs at once:
- **Environment manager** — creates and manages the isolated environment folders.
- **Package manager** — installs software into them.

Mental picture:

```
Your computer
├── system Python              (leave alone — the OS may depend on it)
├── conda envs/
│   ├── scrna/    → python 3.11, numpy 2.x, pandas, scanpy...   ← our project
│   ├── projectB/ → python 3.10, numpy 1.24, tensorflow...
│   └── base/     → keep minimal; don't work in here
```

## 2. Why it exists (why conda, for bioinformatics specifically)

- **Manages non-Python software.** Bio packages depend on compiled C/C++/Fortran libraries (BLAS, HDF5, igraph). Python's own installer `pip` can't install those system binaries reliably; conda can, because it packages them.
- **Reproducibility.** The exact version of every package can be frozen to a file and rebuilt later or by a colleague — the foundation of reproducible science.
- **The bioconda ecosystem.** A large catalog of biology tools is pre-packaged for conda; it's the clean way to get non-pure-Python tools later.

## 3. When it is used

At the **very start** of a project, before writing any analysis code. Create the environment once, then **activate it at the start of every working session** and run everything inside it. Rule: **one environment per project.**

## 4. How it works internally

- An environment is physically just a directory containing a `python` executable and a `site-packages` folder for installed libraries.
- **Activating** an environment prepends that directory to the shell's `PATH`, so `python` / `import numpy` resolve to *that environment's* versions first. Deactivating reverts `PATH`. No magic — directory precedence.
- Installing consults **channels** (online package repos). Community standard = **conda-forge**; biology = **bioconda**. Conda then runs a **dependency solver** (a constraint-satisfaction problem: find versions where all requirements hold) before downloading — this is the pause you see on install.

## 5. Common mistakes

- **Working in `base`.** It's conda's own home; polluting it can break conda itself. Always make a named project environment.
- **`sudo pip install` into system Python.** Corrupts the OS Python. Never.
- **Carelessly mixing `conda` and `pip`.** pip ignores conda's solver and can overwrite conda-managed files → silent breakage. If both are needed: install everything possible with conda **first**, then pip for leftovers, and don't re-run conda after.
- **Not pinning the Python version** at creation → inconsistent behavior across machines.
- **Forgetting to activate.** Installing into `scrna` but running from `base` → imports fail. Check the active environment first when imports mysteriously break.

## 6. Best practices

- One environment per project; keep `base` empty.
- Pin the Python version at creation.
- Use **conda-forge** as the primary channel.
- Export the environment to a file and keep it with the project (reproducibility).
- Modern conda has a fast solver by default; `mamba` is a drop-in faster alternative if needed.

## 7. Alternatives (and why not, for this project)

| Tool | Note | Verdict here |
|---|---|---|
| `venv` + `pip` (built-in) | Lightweight, pure-Python only | Can't install bio binaries → **no** |
| Poetry / pipenv | Great dependency managers, still pip-based | Same binary limitation → **no** |
| Docker / containers | Gold standard reproducibility, heavier | Later-module concern |
| pixi / uv | Newer, very fast, gaining traction | Ecosystem/tutorials assume conda → learn mainstream first |

**Choice:** conda uniquely covers isolation + non-Python binaries + the biology package ecosystem + reproducibility — the exact intersection this project needs.

## 8. Hands-on

Install **Miniforge** (minimal conda installer, defaults to conda-forge) from its official source (`github.com/conda-forge/miniforge`) if conda isn't already present — chosen over full Anaconda because it's lean and channel-clean.

```bash
# 1. Create an isolated environment, pin Python, install the core stack from conda-forge
conda create -n scrna -c conda-forge python=3.11 numpy pandas matplotlib scipy anndata -y

# 2. Activate it — run this at the start of EVERY session
conda activate scrna

# 3. Confirm you're inside it and Python is the right version
python --version   # expect 3.11.x
```

- `conda create` builds a new environment; `-n scrna` names it; `-c conda-forge` sets the channel; `python=3.11` pins the interpreter; the package list is the prerequisite stack (`anndata` pulls `numpy`/`pandas`/`scipy` as dependencies, but naming them is clearer); `-y` auto-confirms.
- `conda activate scrna` switches the shell into the environment (the `PATH` mechanism from §4); the prompt usually shows `(scrna)`.
- `python --version` confirms you're on the environment's Python, not system Python.

> Jupyter and scanpy are intentionally **not** installed yet — Jupyter is the next topic; scanpy belongs to Module 1.

## 9. One-line summary

A virtual environment is an isolated folder with its own Python and packages; conda creates these environments and installs packages (including non-Python binaries) from channels like conda-forge, resolving versions with a solver — so keep `base` clean, pin Python, use conda-forge, and run one environment per project.
